In [1]:
from model_engine.assets.utils import load_asset, get_default_factory_configs

from model_engine.model_builder.asset_parser import asset_parser
from model_engine.client_predictor.client_predictor import ClientPredictor
from model_engine.assets.power.app_asset import schema
from model_engine.model_builder import ModelBuilder
from model_engine.io.utils import global_load_data, global_archive_data, global_path_exists
from model_engine.power.post_sale.base.utils import normalize_io
from model_engine.power.post_sale.base import ClientHandler, ModelConstraints, PowerModelingBase, S3ClientHandler
from model_engine.power.post_sale import ClientModelBuilder
from model_engine.io.loaders import DictHelper
from model_engine.io.s3_schema_manager import S3SchemaManager
import s3fs
import pandas as pd
from zestio import DataArchiver
import copy
from projectz import ZInfo
from zestio import handler
from model_engine.io.loaders import data_loader
from zestio.handler import FileSystemHandler, S3Handler
import json
import os
from zestio import handler
from zaml.common.utils.io import load_state
from zestio import handler

/home/jag/.local/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources



In [2]:
def connect_client(z_info, client_name):
    z_info.connect_client(name = client_name)
    return z_info
def connect_datasets(z_info, dataset_version):
    datasets = z_info.context.web_handler.get(f"clients/{z_info.client.id}/datasets")
    dataset_num = dataset_version-1
    dataset_model_type = datasets[dataset_num]['model_type']
    dataset_data_source = datasets[dataset_num]['data_source']
    dataset_product = datasets[dataset_num]['product']
    dataset_version = datasets[dataset_num]['version']
    z_info.connect_dataset(model_type = dataset_model_type,
                                     product = dataset_product, 
                                     version = dataset_version)
    return z_info
def connect_model(z_info, model_name, product):
    z_info.connect_model_iteration(name = model_name, product = product,  model_type=['underwriting'])
    return z_info
def connect_to_z_info(dataset_version, product, model_name, client_name):
    z_info = ZInfo()
    z_info =connect_client(z_info, client_name)
    z_info = connect_datasets(z_info, dataset_version)
    z_info = connect_model(z_info, model_name, product)
    return z_info
def get_alternate_paths(processed_path):
    """Takes a PROCESSED path and returns PREPROCESSED and NORMALIZED paths"""
    # PROCESSED:    TABLE=X/VERSION/CLIENT/PRODUCT/BUREAU/FORMAT/PULL_DATE/PULL_NAME/ME_VERSION
    # PREPROCESSED: BUREAU/FORMAT/TABLE=X/VERSION/CLIENT/PRODUCT/PULL_DATE/PULL_NAME/ME_VERSION  
    # NORMALIZED:   TABLE=X/VERSION/CLIENT/PRODUCT/BUREAU/FORMAT (no PULL_DATE onwards? - check screenshot)
    
    # Extract partitions from processed path
    parts = processed_path.replace('s3://power-client-data-staging/PROCESSED/DATA/', '').rstrip('/').split('/')
    partitions = {}
    for p in parts:
        if '=' in p:
            key, val = p.split('=', 1)
            partitions[key] = val
    
    base = 's3://power-client-data-staging'
    
    preprocessed = (f"{base}/PREPROCESSED/DATA/"
                    f"BUREAU={partitions['BUREAU']}/FORMAT={partitions['FORMAT']}/"
                    f"TABLE={partitions['TABLE']}/VERSION={partitions['VERSION']}/"
                    f"CLIENT={partitions['CLIENT']}/PRODUCT={partitions['PRODUCT']}/"
                    f"PULL_DATE={partitions['PULL_DATE']}/PULL_NAME={partitions['PULL_NAME']}/"
                    f"ME_VERSION={partitions['ME_VERSION']}/")
    
    normalized = (f"{base}/NORMALIZED/DATA/"
                  f"TABLE={partitions['TABLE']}/VERSION={partitions['VERSION']}/"
                  f"CLIENT={partitions['CLIENT']}/PRODUCT={partitions['PRODUCT']}/"
                  f"BUREAU={partitions['BUREAU']}/FORMAT={partitions['FORMAT']}/")
    
    return preprocessed, normalized

In [3]:
dataset_version = 1
product = ['autoloan']
model_name = 'model1_AppData_LTVAutoloan'
client_name = 'californiacu'

z_info = connect_to_z_info(dataset_version, product, model_name, client_name)

INFO:projectz.logger:Connected client: californiacu
INFO:projectz.logger:Connected client: californiacu
INFO:projectz.logger:Connected to client: californiacu
INFO:projectz.logger:Connected dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected to dataset: autoloanCreditcardHomeequityPersonalloanv1
INFO:projectz.logger:Connected model_iteration: autoloan/model1_AppData_LTVAutoloan
INFO:projectz.logger:Connected model_iteration: autoloan/model1_AppData_LTVAutoloan
INFO:projectz.logger:Connected to model_iteration: autoloan/model1_AppData_LTVAutoloan


In [4]:
model_artifacts_dir = z_info.model_iteration._dir_paths['modeling_model_artifacts_dir']

# The Asset contains the paths to our directory

In [5]:
asset = load_asset(os.path.join(model_artifacts_dir, 'asset.json'))

In [6]:
os.listdir(os.path.join(model_artifacts_dir,'fe_data'))

['split=test', 'split=train']

In [7]:
os.listdir(os.path.join(model_artifacts_dir, 'fe_data', 'split=test'))

['data=client']

In [8]:
fe_data = pd.read_parquet(os.path.join(model_artifacts_dir, 'fe_data', 'split=test','data=client'))

In [9]:
trade_columns = ['ZEST_KEY'] + [col for col in fe_data.columns if col.startswith('trade')] 

In [10]:
fe_data[trade_columns]

,ZEST_KEY,trade_max_balance_to_hc__with_historic_utilization_over_50_percent_revolving,trade_count__non_derog_closed_accounts,trade_max_credit_limit__with_utilization_over_25_percent_open_cc,trade_count__with_recent_payment_open_accounts,trade_count__non_derog_accounts,trade_mean_historic_utilization__with_recent_payment_open_cc,trade_percent_derog__all_accounts,trade_percent_accounts_with_DQ30_or_greater_in_last_6_months__with_recent_payment_open_accounts,trade_mean_percent_unpaid__with_recent_payment_personal,...,trade_mean_balance_to_hc__active_open_revolving,trade_mean_historic_utilization__non_derog_open_cc,trade_sum_loan_amount__never_delinquent_closed_installment,trade_sum_balance__unsecure_open_accounts,trade_max_historic_utilization__with_utilization_over_50_percent_revolving,trade_percent_recently_open__with_historic_utilization_over_100_percent_cc,trade_months_since_oldest_account_opened__secure_accounts,trade_count__non_derog_closed_unsecure_personal,trade_max_utilization__non_derog_open_revolving,trade_max_past_due_to_monthly_payment__all_closed_auto
0,303683_1_276_12,0.687027,0.0,-3.0,2.0,4.0,-3.000000,0.428571,0.0,-3.000000,...,0.843513,0.088000,-2.0,2900.0,0.909556,-3.0,25.856794,0.0,0.624889,-2.0
1,303687_1_276_12,-2.000000,0.0,-2.0,0.0,0.0,-2.000000,1.000000,-2.0,-2.000000,...,-2.000000,-2.000000,-2.0,-2.0,-2.000000,-2.0,-3.000000,0.0,-2.000000,-2.0
2,303691_1_276_12,0.887457,6.0,10000.0,5.0,13.0,0.923969,0.000000,0.0,0.744103,...,0.544727,0.923969,6615.0,11847.0,0.956692,-3.0,107.566890,1.0,0.849023,-2.0
3,303705_2_276_12,0.569787,14.0,15000.0,5.0,20.0,0.807408,0.000000,0.0,0.872617,...,0.519402,0.807408,-2.0,5395.0,-3.000000,0.0,234.616727,0.0,0.482500,-2.0
4,303705_1_276_12,-1.000000,-1.0,-1.0,-1.0,-1.0,-1.000000,-1.000000,-1.0,-1.000000,...,-1.000000,-1.000000,-1.0,-1.0,-1.000000,-1.0,-1.000000,-1.0,-1.000000,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12288,344653_2_276_22,-3.000000,0.0,-3.0,1.0,2.0,0.243000,0.000000,0.0,-2.000000,...,0.010288,0.243000,-2.0,4760.0,-3.000000,-3.0,-3.000000,0.0,0.005000,-2.0
12289,344654_2_276_22,0.000000,24.0,-3.0,3.0,31.0,0.245000,0.000000,0.0,-2.000000,...,0.028734,0.155121,8580.0,721.0,-3.000000,0.0,69.093821,0.0,0.048067,-4.0
12290,344654_1_276_22,0.000000,8.0,-2.0,1.0,10.0,-2.000000,0.000000,0.0,-2.000000,...,-2.000000,-2.000000,39303.0,-3.0,-3.000000,-3.0,149.883981,0.0,-2.000000,-4.0
12291,344655_1_276_22,0.758442,0.0,2500.0,1.0,2.0,0.924000,0.000000,0.0,-2.000000,...,0.379221,0.561250,-2.0,1752.0,0.924000,-3.0,-3.000000,0.0,0.700800,-2.0


In [11]:
from model_engine.io.loaders import data_loader
trade_data = data_loader(asset['data']['trade']['data'], asset={'info': {'key': 'ZEST_KEY'}}).reset_index()[trade_columns]


severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.




# We can see that the data is the same for people in our FE who have tradelines

In [12]:
merged_trade_data = trade_data.merge(fe_data[['ZEST_KEY']], on='ZEST_KEY', how='inner')
merged_fe_data_trade_columns = fe_data[trade_columns].merge(trade_data[['ZEST_KEY']], on = 'ZEST_KEY', how = 'inner')

# For people who have a tradeline it is the same as the processed data

In [13]:
import numpy as np
np.unique(merged_trade_data == merged_fe_data_trade_columns)

array([ True])

# We can find the original preprocessed data

In [14]:
processed_trade = asset['data']['trade']['data'][0]

In [15]:
preprocessed_path, normalized_path = get_alternate_paths(processed_trade)

# Let's subset preprocessed to only have 500 of our members

In [36]:
members_to_select = fe_data.reset_index()[['ZEST_KEY']][0:500]

In [38]:
preprocessed_path

's3://power-client-data-staging/PREPROCESSED/DATA/BUREAU=equifax/FORMAT=cms_6/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'

In [39]:
asset['data']['trade']['data'][0]

's3://power-client-data-staging/PROCESSED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/PULL_DATE=2026-02-01/PULL_NAME=20260131_allin_californiacu_downeyfcu_penair_safe1_sesloccu_trumarkfinancialcu_vermontfcu/ME_VERSION=v2.1.1/'

In [40]:
normalized_path

's3://power-client-data-staging/NORMALIZED/DATA/TABLE=trade/VERSION=v2/CLIENT=californiacu/PRODUCT=autoloan/BUREAU=equifax/FORMAT=cms_6/'

In [41]:
all_preprocessed = pd.read_parquet(preprocessed_path)

In [42]:
preprocessed = all_preprocessed.merge(members_to_select, on = 'ZEST_KEY', how = 'inner')

In [43]:
members_to_select

,ZEST_KEY
0,303683_1_276_12
1,303687_1_276_12
2,303691_1_276_12
3,303705_2_276_12
4,303705_1_276_12
...,...
495,305370_1_276_12
496,305371_1_276_12
497,305375_1_276_12
498,305378_1_276_12


In [44]:
all_preprocessed

,ZEST_KEY,quarter,appDate,SEG_SEQ,SEG_PARENT,SEG_PARENT_SEQ,SEGMENT_TYPE,CUSTOMER_NAME,CUSTOMER_NUMBER,DATE_REPORTED,...,PREVIOUS_HIGH_DATE_BEFORE_HISTORY,NARRATIVE_CODE_1X,NARRATIVE_CODE_2X,NARRATIVE_CODE_3X,NARRATIVE_CODE_4X,RECENT_TRADE_FLAG,QUALIFYING_FLAG,ARCHIVE_DATA,DATE_OF_REQUEST,ARCHIVE_DATE
0,210623_1_276_12,2021Q2,2021-07-01,1,FULL-Header,1,PT,None,BB,06232021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
1,210623_1_276_12,2021Q2,2021-07-01,2,FULL-Header,2,PT,None,BB,06162021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
2,210623_1_276_12,2021Q2,2021-07-01,3,FULL-Header,3,PT,None,OC,05212021,...,None,166,None,None,None,Y,N,None,2021-06-29,2021-06-30
3,210623_1_276_12,2021Q2,2021-07-01,4,FULL-Header,4,PT,None,ON,05172021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
4,210623_1_276_12,2021Q2,2021-07-01,5,FULL-Header,5,PT,None,ON,04212021,...,None,None,None,None,None,Y,Y,None,2021-06-29,2021-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1845433,531912_2_276_22,2025Q3,2025-12-31,10,FULL-Header,10,PT,None,FM,11062020,...,None,208,132,None,None,Y,Y,None,2025-09-30,2025-09-30
1845434,531912_2_276_22,2025Q3,2025-12-31,11,FULL-Header,11,PT,None,CG,07282018,...,None,065,None,None,None,Y,Y,None,2025-09-30,2025-09-30
1845435,531912_2_276_22,2025Q3,2025-12-31,12,FULL-Header,12,PT,None,FA,01312018,...,None,None,None,None,None,Y,Y,None,2025-09-30,2025-09-30
1845436,531912_2_276_22,2025Q3,2025-12-31,13,FULL-Header,13,PT,None,BB,12312017,...,None,None,None,None,None,Y,Y,None,2025-09-30,2025-09-30


In [45]:
preprocessed.shape

(9723, 66)

# How did we convert from preprocessed to normalized?

In [46]:
asset['data']['trade']['asset']

{'fe_version': 2,
 'InputNormalizer': 'equifax/cms_6/fe2/trade.json',
 'AggregationEngine': 'aggregation/fe2/trade.json'}

# We can see that for our InputNormalizer we used the asset `equifax/cms_6/fe2/trade.json`

* The InputNormalizer asset contains the mapping and preprocessing instructions for taking raw bureau data (columns like `BALANCE`, `DATE_OPENED`, `CREDIT_LIMIT`) and converting them to canonical column names and intermediate features that the AggregationEngine expects
* The goal of normalization is to get different bureaus (Equifax, Experian, Transunion) into the same standardized format - so we can apply one AggregationEngine to all of them regardless of which bureau the data came from
* This is what runs in production when the data arrives only preprocessed - in our training case the data is already processed so the FE engine passes through, but the asset is still included in the pipeline so it works in both contexts

## Thus, if the asset were a map, the InputNormalizer is the set of directions for how to get from raw bureau columns to a standardized format

In [47]:
asset_trade_normalizer = load_asset(asset['data']['trade']['asset']['InputNormalizer'])

In [48]:
pipeline = load_state(os.path.join(model_artifacts_dir, 'pipeline.obj'))

                      but the loaded object is built on ZAML 34.5.4. Loading serialized
                      object from different versions are not recommended and may yield unexpected
                      errors or results.

                      (name, curr_ver, org_ver): [('ipywidgets', '8.1.8', '7.8.5'), ('jupyter-book', '1.0.4.post1', '0.15.1'), ('pyarrow_hotfix', '0.6', '0.7'), ('scikit-learn', '1.2.2', '1.2.1'), ('Sphinx', '7.4.7', '5.0.2'), ('sphinx-rtd-theme', '3.1.0', '1.3.0'), ('sphinxcontrib-bibtex', '2.6.1', '2.5.0'), ('xlrd', '2.0.1', '2.0.2')]. This might yield unexpected result if the object relies
                      on this package.



In [49]:
node_dict = {}
for node in pipeline.graph.nodes:
    print(node.name)
    node_dict[node.name] = node
    

app
bnkr
client_data
collec
config
inq
member
trade
app FE
bnkr FE
client_data FE
collec FE
config FE
inq FE
member FE
trade FE
Merge data
Bivariate Feature Engineering
Drop Features
OneHotEncoder
FillNA
GeneralFeatureSelection
Drop Engineered Features


# We can see that our trade feature transformer is an EndToEndFeatureEngine

In [50]:
node_dict['trade FE'].transformer

# Furthermore,  we can see that it again points to the same parts of the asset: the normalizer and the aggregation engine

In [51]:
node_dict['trade FE'].transformer.asset.keys()

dict_keys(['fe_version', 'InputNormalizer', 'AggregationEngine', 'table_name', 'info'])

## We can see that the InputNormalizer Asset here is the same as the asset we had before

In [52]:
node_dict['trade FE'].transformer.asset['InputNormalizer']

{'info': {'key': 'ZEST_KEY', 'reference_date': 'DATE_OF_REQUEST'},
 'mapping': [{'type': 'StringConverterV2',
   'params': {'raw_feature': 'ZEST_KEY'}},
  {'type': 'DateConverterV2',
   'params': {'raw_feature': 'DATE_OF_REQUEST',
    'new_feature': 'date_of_request'}},
  {'type': 'NumericConverterV2',
   'params': {'raw_feature': 'BALANCE',
    'new_feature': 'balance_amt',
    'fillna': 0,
    'fillblanks': 0}},
  {'type': 'NumericConverterV2',
   'params': {'raw_feature': 'CREDIT_LIMIT', 'new_feature': 'credit_limit'}},
  {'type': 'NumericConverterV2',
   'params': {'raw_feature': 'HIGH_CREDIT', 'new_feature': 'high_credit_amt'}},
  {'type': 'NumericConverterV2',
   'params': {'raw_feature': 'PAST_DUE_AMOUNT',
    'new_feature': 'pastDueAmt',
    'fillna': 0,
    'fillblanks': 0}},
  {'type': 'NumericConverterV2',
   'params': {'raw_feature': 'SCHEDULED_PAYMENT_AMOUNT',
    'new_feature': 'scheduled_payment_amount',
    'fillna': 0,
    'fillblanks': 0},
   'drop_from_normalized': T

In [53]:
node_dict['trade FE'].transformer.asset['InputNormalizer'] == asset_trade_normalizer

True

# We can see inside the transformer it has two transformer: an InputNormalizer and an Aggregation Engine

In [54]:
node_dict['trade FE'].transformer.transformers

OrderedDict([('InputNormalizer',
              <model_engine.feature_engine_V2.feature_engine.InputNormalizer at 0x7fbd41ddc790>),
             ('AggregationEngine',
              <model_engine.feature_engine_V2.feature_engine.AggregationEngine at 0x7fbd41dde1d0>)])

In [55]:
node_dict['trade FE'].transformer.transformers['InputNormalizer']

## Thus, if the InputNormalizer asset is the set of directions, then the `InputNormalizer` object is the driver that actually follows those directions - it reads the asset's mapping and preprocessing instructions and executes the transformations on the data

# We can see the source code for the InputNormalizer object [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/feature_engine_V2/feature_engine.py#L149)

* It has two main operations: **mapping** (converts raw bureau columns to canonical names and types - e.g., `BALANCE` -> `balance_amt` as numeric, `DATE_OPENED` -> `openDate` as date, `TERMS_FREQUENCY` -> `termFreqMult` with a frequency-to-multiplier mapping) and **preprocessing** (creates derived intermediate columns and binary filter flags from the mapped data - e.g., `months_since_openDate` from date diffs, `is_open`/`is_closed`/`is_revolving`/`is_auto` from account codes and narrative codes, `crdUtl` from balance/credit_limit ratios, and delinquency status flags from payment history patterns)
* The goal of normalization is to get different bureaus (Equifax, Experian, Transunion) into the same standardized format - so we can apply one AggregationEngine to all of them regardless of which bureau the data came from
* The output is row-level normalized data (still one row per tradeline) with standardized column names and all the filter flags the AggregationEngine needs to compute its aggregations

# We can see the `transform` method checks if the output features already exist in the data - if they do, it just returns them without running any transformations. This is the pass-through behavior that lets the same pipeline work with both processed data (development) and preprocessed data (production).

In [56]:
import inspect

print(inspect.getsource(node_dict['trade FE'].transformer.transformers['InputNormalizer'].transform))

    def transform(self, data, small_batch=False):
        '''
        Base transformation function. 

        Validates if data has all expected incoming features, and if so calls transform() on each transformer of the stack.

        Parameters
        ----------
        data : DataFrame
            Input data to transform.
            If all expected incoming feature names are present in data, just returns those features from data without additional transformation

        small_batch : bool, default=False
            Whether to use small batch processing for transformers that support it 

        Returns
        -------
        DataFrame
            Transformed data with output features

        Raises
        ------
        FeatureEngineAssetError
            If required input features are missing from the data
        '''
        incoming_features = set(data.columns)
        output_features = set(self.keep_features)
        expected_incoming_features = set(self.input_features)

  

In [57]:
preprocessed.head(5)

,ZEST_KEY,quarter,appDate,SEG_SEQ,SEG_PARENT,SEG_PARENT_SEQ,SEGMENT_TYPE,CUSTOMER_NAME,CUSTOMER_NUMBER,DATE_REPORTED,...,PREVIOUS_HIGH_DATE_BEFORE_HISTORY,NARRATIVE_CODE_1X,NARRATIVE_CODE_2X,NARRATIVE_CODE_3X,NARRATIVE_CODE_4X,RECENT_TRADE_FLAG,QUALIFYING_FLAG,ARCHIVE_DATA,DATE_OF_REQUEST,ARCHIVE_DATE
0,303683_1_276_12,2023Q1,2023-04-21,1,FULL-Header,1,PT,None,DC,03202023,...,None,None,None,None,None,Y,Y,None,2023-03-28,2023-03-31
1,303683_1_276_12,2023Q1,2023-04-21,2,FULL-Header,2,PT,None,DC,03202023,...,None,None,None,None,None,Y,Y,None,2023-03-28,2023-03-31
2,303683_1_276_12,2023Q1,2023-04-21,3,FULL-Header,3,PT,None,FY,03032023,...,None,057,None,None,None,Y,Y,None,2023-03-28,2023-03-31
3,303683_1_276_12,2023Q1,2023-04-21,4,FULL-Header,4,PT,None,FA,02282023,...,None,132,None,None,None,Y,Y,None,2023-03-28,2023-03-31
4,303683_1_276_12,2023Q1,2023-04-21,5,FULL-Header,5,PT,None,ON,03032023,...,None,None,None,None,None,Y,Y,None,2023-03-28,2023-03-31


In [58]:
asset_trade_normalizer.keys()

dict_keys(['info', 'mapping', 'preprocess', 'drop_columns', 'table_type', 'validator', 'feature_engineering', 'fe_version', 'etl_format', 'table_name', 'prod_etl'])

## `info.key = 'ZEST_KEY'` tells the normalizer which column to use as the join key across all tables
## `info.reference_date = 'DATE_OF_REQUEST'` is the anchor date that all time-based calculations are relative to - for example, `months_since_openDate` is the number of months between when the account was opened and when the bureau data was pulled

In [59]:
asset_trade_normalizer['info']

{'key': 'ZEST_KEY', 'reference_date': 'DATE_OF_REQUEST'}

In [60]:
type(asset_trade_normalizer['mapping'])

list

## `mapping` is a list of converter operations that each take a single raw Equifax column and convert it to a standardized name and type - `NumericConverterV2` for numbers (`BALANCE` -> `balance_amt`), `DateConverterV2` for dates (`DATE_OPENED` -> `openDate`), and `StringConverterV2` for strings (`ACCOUNT_TYPE` -> `accountType`). Some also apply value mappings (like `TERMS_FREQUENCY` -> `termFreqMult` where `'M'` becomes `1`, `'W'` becomes `4.33`, etc.)
## Columns marked `drop_from_normalized: True` are intermediate helpers that get used in preprocessing but won't appear in the final normalized output - for example, `PAYMENT_HISTORY_1_24` is needed to compute delinquency patterns but gets dropped after

## high_credit_amt: The highest reported balance for that tradeline
* Installment accounts: if you haven't missed any payments, this should be the balance you started with. If you miss payments, this can accrue on the high credit
* Revolving accounts: the highest balance you had at that point, even if you pay off immediately it would be the most expensive purchase
* See [App Review Guide](https://zestfinance.atlassian.net/wiki/spaces/DS/pages/1710784539/App+Review+Guide) for more details
* We apply a NumericConverter from `HIGH_CREDIT`

## balance_amt: The amount of debt you have right now
* We apply NumericConverter from `BALANCE`, filling NA with 0

## credit_limit: The maximum amount you can borrow
* We first apply a NumericConverter from `CREDIT_LIMIT`, then later in preprocessing we set it to NA for non-revolving, non-open, and non-charge card accounts
* Only makes sense in the context of revolving accounts

## pastDueAmt: The dollar amount on the tradeline that is past due at the report date (may include fees and interest)
* The amount of money you should have paid by now but have not yet
* Revolving: this can be the minimum payments you have missed so far
* Installment: generally equals the sum of the missed payments
* We create from `PAST_DUE_AMOUNT`, filling NA with 0

## scheduled_payment_amount: Contractual amount due for next payment
* Installment: the fixed payment
* Revolving: the minimum payment amount
* We create with NumericConverter from `SCHEDULED_PAYMENT_AMOUNT`
* We later multiply by `termFreqMult` to get `monthly_payment_amt`

## termDur: How long the payment lasts
* The amount of time to repay the loan (page 271 in Equifax docs)
* We create from `TERMS_DURATION`, we do not fill NA with 0 here

## termFreqStr: String version of how often you have to pay

## termFreqMult: How often you have to pay, as a monthly multiplier
* We create from `TERMS_FREQUENCY` using a NumericConverter with a mapping that converts frequency codes into how many months there are per payment period:
```
"M": 1, "B": 2.17, "W": 4.33, "E": 2, "L": 0.5, "Q": 0.33, "S": 0.17, "T": 0.25, "Y": 0.08, "D": 1, "P": 1
```
* Missing values default to 1 (monthly). See page 155 in the Equifax document

## date_of_request: The application date
* Used as the reference date for all time-based calculations. For national data, each person gets a random application date between 0 and 3 months after the latest reported date for their tradelines

## openDate: Date that tradeline was opened
* Created with DateConverter from `DATE_OPENED`

## closedDate: Date that the tradeline was closed
* Equifax: "contains the date the account was closed. It will not be populated when Date Major Delinquency 1st Reported is present"
* Created from `CLOSED_DATE`

## majordqDate: Date of first major delinquency
* Populated if current rate/status is 6 (collection), 7 (chapter 13 bankruptcy), 8 (repossession), 9 (charge off), M, or Z (foreclosure)
* Created from `DMD_REPORTED`

## rptDate: Date the tradeline was last updated

## lstPmtDate: Date the user made the last payment
* Created with DateConverter from `LAST_PAYMENT_DATE`

## accountType: Type of loan
* Created from `ACCOUNT_TYPE` using StringConverter
* See page 150 in the Equifax document for all codes (e.g., 18 = credit card)
* "Contains a code that describes the kind of loan (auto, home improvement, credit card, etc)"

## portfolioType: The type of loan (more general)
* Created from `PORTFOLIO_TYPE` using StringConverter, mapping `O` (open-ended) to `R` (revolving). Open is typically a charge card where you have full payment every cycle

## PORTFOLIO_TYPE: The type of loan (with open kept separate)
* Same column without the O -> R mapping, kept as an intermediate so filters like `is_revolving` and `is_open_ended` can distinguish between them

## ecoa: Relationship of the person to the tradeline
* Created from `ECOA_DESIGNATOR` using StringConverter
* In the preprocessing step, we drop rows where ecoa is `A` (authorized user), `T` (terminated), or `X` (deceased) since these people are not financially responsible for the tradeline

### Account Designator Codes (ECOA)

| CODE | DESCRIPTION |
| :---: | :--- |
| A | Authorized User - another individual has contractual responsibility |
| B | On behalf of another person - the subject has financial responsibility but the account is used exclusively by another person |
| C | Co-maker - the subject has co-signed for a loan and will be responsible if the borrower defaults |
| I | Individual Account - the subject is primarily responsible for payment |
| J | Joint Account - the subject and another person are jointly responsible for payment |
| M | Maker - the subject is responsible for payment, but a co-maker will be responsible if maker defaults |
| S | Shared, but otherwise undesignated - credit grantor knows the subject and at least one other person share the account but not enough info to designate as J or A |
| T | Terminated - the subject's relationship to this account has ended |
| U | Undesignated |
| X | Deceased (not returned on trade lines) |

## NARRATIVE_CODE_1 and NARRATIVE_CODE_2
* Both created using StringConverter
* These are comment codes that indicate certain conditions about the tradeline (e.g., `BZ` = account paid for less than full balance, `CB` = charged off, `IL` = chapter 7 bankruptcy, `DO` = chapter 13 bankruptcy)
* Both codes can carry the same types of values - a tradeline can have up to two narrative codes at once
* These are used heavily in the preprocessing filters (e.g., `is_bnkr`, `is_closed`, `is_chargeoff`, `is_negative`) to identify account statuses that the `RATE_STATUS_CODE` alone doesn't capture

## dqDate: Date of the highest delinquency that occurred outside the payment history window
* Created from `PREVIOUS_HIGH_DATE_1` using DateConverter
* The payment history covers the most recent 48 months. `dqDate` captures any delinquency that happened before that window
* Used in the `is_never_dq` filter - if `dqDate` is NA and there are no delinquencies in the 48-month payment history, the tradeline has never been delinquent

## RATE_STATUS_CODE: The current rating on the account
* Rate codes are numeric, status codes are letters
* Created using StringConverter, used as an intermediate (dropped from normalized output)

### Rate Codes

| CODE | DESCRIPTION |
| :---: | :--- |
| 0 | Too new to rate; approved but not used |
| 1 | Pays account as agreed |
| 2 | Not more than two payments past due (DQ30) |
| 3 | Not more than three payments past due (DQ60) |
| 4 | Not more than four payments past due (DQ90) |
| 5 | At least 120 days or more than four payments past due (DQ120) |
| 6 | Collection account |
| 7 | Included in Chapter 13 |
| 8 | Repossession |
| 9 | Charge-off |

### Status Codes

| CODE | DESCRIPTION |
| :---: | :--- |
| A | Account is inactive |
| B | Lost or stolen card |
| C | Contact member for status |
| D | Refinanced or renewed |
| E | Consumer deceased |
| G | Foreclosure process started |
| M | Included in Chapter 13 |
| S | Dispute - resolution pending |
| Z | Included in Bankruptcy |
| # | In bankruptcy of another person (retired 2009) |

## Payment_History_1_24, Payment_History_25_36, Payment_History_37_48
* The payment history strings covering the most recent 48 months. Leftmost character is most recent month
* Created using StringConverter, used as intermediates (dropped from normalized output)
* The `PaymentPatternsAggregatorV2` preprocessor combines all three strings, trims to 48 months, and extracts trended features like `number_DQ30+_24_months`, `number_CO_12_months`, and `months_since_DQ60+`

## ACTIVITY_DESIGNATOR: The final state of the account
* Created using StringConverter, used as an intermediate (dropped from normalized output)

| Code | Description |
| :---: | :--- |
| B | Paid and Closed |
| C | Closed |
| D | Transfer/Sold/Paid |
| L | Lost/Stolen |
| P | Paid |
| R | Refinanced |
| T | Transfer/Sold |

* Used in the `is_closed` and `is_transfer` filters

## chargeoff_amt: The amount the creditor wrote off when the account was charged off
* Created from `ORIGINAL_CHARGE_OFF_AMOUNT` using NumericConverter
* In preprocessing, if `high_credit_amt` is NA we replace it with `chargeoff_amt` via the `CoalesceV2` step - our best guess for the original loan amount when the high credit is missing

## CUSTOMER_NUMBER: The industry code / kind of business
* Created using StringConverter, used as an intermediate (dropped from normalized output)
* Used to create the `is_member` filter where `CUSTOMER_NUMBER == 'FC'` identifies credit union tradelines

## constant_placeholder: A constant column of 1s for every row
* Created from `ZEST_KEY` using StringConverter with `replace_all_with_constant: '1'`
* Used in the AggregationEngine for count aggregations so that null values in other columns don't affect the count



## We can see that our InputNormalizer has the mapper transformer, preprocess transformer, and the post process transformer

In [61]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers

OrderedDict([('Mappers',
              <model_engine.feature_engine_V2.listed_objects_engines.MapperV2 at 0x7fbd41ddc7c0>),
             ('Preprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fbd41ddcd90>),
             ('Postprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fbd41dde140>)])

## We can see our Mapper is a [MapperV2](https://github.com/Katlean/model-engine/blob/master/model_engine/feature_engine_V2/listed_objects_engines.py#L135) object which takes in the mapping asset as an input

It is called [here](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/feature_engine_V2/feature_engine.py#L186) in InputNormalizer
```python
mapper = MapperV2(api=self.asset['mapping'])
```

* `MapperV2` takes the mapping list from the asset and converts each dict into an actual transformer object (e.g., `NumericConverterV2`, `DateConverterV2`, `StringConverterV2`) stored in an `OrderedDict` - the order matters because each transformer's output is fed as input to the next
* When `transform(data)` is called, it loops through each transformer in order and applies it to the data, progressively renaming and converting raw bureau columns into their canonical forms

In [62]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Mappers']

In [68]:
node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers

OrderedDict([('Mappers',
              <model_engine.feature_engine_V2.listed_objects_engines.MapperV2 at 0x7fbd41ddc7c0>),
             ('Preprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fbd41ddcd90>),
             ('Postprocessors',
              <model_engine.feature_engine_V2.listed_objects_engines.PreprocessorV2 at 0x7fbd41dde140>)])

## These individual transformers (NumericConverterV2, DateConverterV2, FilterV2, etc.) live in the [feature-engine-parts](https://github.com/Katlean/feature-engine-parts/tree/master/feature_engine_parts/fe_parts_V2) codebase

* If the asset is the blueprint and the InputNormalizer is the foreman, then `feature-engine-parts` are the actual tools and workers - each converter, filter, and aggregator is a lightweight transformer class that does one specific job. As [Jackie put it](https://drive.google.com/file/d/1cTEUrDFCauPTh2Fhs5hYMoiFQ1IKl9Me/view): "feature-engine-parts are all the smaller bricks, model-engine is the design that tells you how to use those bricks to build a house"


In [64]:
mapper_dict = {}
for mapper in node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Mappers'].stack:
    name = mapper.transformer_name
    print(f'mapper: {mapper} has transformer: {mapper.transformer_name}')
    print(f'raw_feature input is {mapper.raw_feature}')
    print(f'output is {mapper.new_feature}')
    mapper_dict[name] = mapper


mapper: <feature_engine_parts.fe_parts_V2.mappers.converters.StringConverterV2 object at 0x7fbd41ddc7f0> has transformer: ZEST_KEY
raw_feature input is ZEST_KEY
output is ZEST_KEY
mapper: <feature_engine_parts.fe_parts_V2.mappers.converters.DateConverterV2 object at 0x7fbd41ddc820> has transformer: date_of_request
raw_feature input is DATE_OF_REQUEST
output is date_of_request
mapper: <feature_engine_parts.fe_parts_V2.mappers.converters.NumericConverterV2 object at 0x7fbd41ddc850> has transformer: balance_amt
raw_feature input is BALANCE
output is balance_amt
mapper: <feature_engine_parts.fe_parts_V2.mappers.converters.NumericConverterV2 object at 0x7fbd41ddc880> has transformer: credit_limit
raw_feature input is CREDIT_LIMIT
output is credit_limit
mapper: <feature_engine_parts.fe_parts_V2.mappers.converters.NumericConverterV2 object at 0x7fbd41ddc8b0> has transformer: high_credit_amt
raw_feature input is HIGH_CREDIT
output is high_credit_amt
mapper: <feature_engine_parts.fe_parts_V2.ma

In [65]:
mapper_dict['PAYMENT_HISTORY_1_24'].__dict__

{'_init_args': (),
 '_init_kwargs': {'raw_feature': 'PAYMENT_HISTORY_1_24'},
 'transformer_name': 'PAYMENT_HISTORY_1_24',
 'raw_feature': 'PAYMENT_HISTORY_1_24',
 'new_feature': 'PAYMENT_HISTORY_1_24',
 'raw_feature_type': 'string',
 'dtype': 'string',
 '_other_params': {},
 'input_dtypes': {'PAYMENT_HISTORY_1_24': 'string'},
 'output_dtypes': {'PAYMENT_HISTORY_1_24': 'string'},
 'pandas_dtype': 'str',
 'strip_whitespace': True,
 'fillna': None,
 'fillblanks': nan,
 'mapping': None,
 'replace_all_with_constant': None,
 'add_params_to_parser_asset': True,
 'level_mapping': None,
 'new_levels': None}

In [66]:
mapped_df = node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Mappers'].transform(preprocessed)

In [67]:
mapped_df

,ZEST_KEY,quarter,appDate,SEG_SEQ,SEG_PARENT,SEG_PARENT_SEQ,SEGMENT_TYPE,CUSTOMER_NAME,CUSTOMER_NUMBER,DATE_REPORTED,...,lstPmtDate,dqDate,termFreqMult,termFreqStr,termDur,accountType,portfolioType,ecoa,chargeoff_amt,constant_placeholder
0,303683_1_276_12,2023Q1,2023-04-21,1,FULL-Header,1,PT,None,DC,03202023,...,2023-02-01,NaT,1.0,M,NaN,07,R,I,NaN,1
1,303683_1_276_12,2023Q1,2023-04-21,2,FULL-Header,2,PT,None,DC,03202023,...,2022-05-01,NaT,1.0,NaN,NaN,06,I,I,NaN,1
2,303683_1_276_12,2023Q1,2023-04-21,3,FULL-Header,3,PT,None,FY,03032023,...,NaT,NaT,1.0,NaN,NaN,0C,R,I,NaN,1
3,303683_1_276_12,2023Q1,2023-04-21,4,FULL-Header,4,PT,None,FA,02282023,...,2023-01-01,NaT,1.0,M,73.0,00,I,I,NaN,1
4,303683_1_276_12,2023Q1,2023-04-21,5,FULL-Header,5,PT,None,ON,03032023,...,NaT,NaT,1.0,M,NaN,18,R,I,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9718,305378_2_276_12,2023Q1,2023-04-29,22,FULL-Header,22,PT,None,FF,09182016,...,2009-10-01,NaT,1.0,NaN,NaN,07,R,J,NaN,1
9719,305378_2_276_12,2023Q1,2023-04-29,23,FULL-Header,23,PT,None,BB,06282016,...,NaT,NaT,1.0,NaN,NaN,18,R,I,NaN,1
9720,305378_2_276_12,2023Q1,2023-04-29,24,FULL-Header,24,PT,None,FM,05072015,...,2015-04-01,NaT,1.0,NaN,360.0,19,M,J,NaN,1
9721,305378_2_276_12,2023Q1,2023-04-29,25,FULL-Header,25,PT,None,BB,08132014,...,NaT,NaT,1.0,NaN,NaN,0G,R,I,NaN,1


# Preprocessing

In [70]:
asset_trade_normalizer['preprocess']

[{'type': 'DateDiffV2',
  'params': {'feature': 'rptDate',
   'reference_feature': 'date_of_request',
   'new_feature': 'months_since_rptDate'}},
 {'type': 'DateDiffV2',
  'params': {'feature': 'openDate',
   'reference_feature': 'date_of_request',
   'new_feature': 'months_since_openDate'}},
 {'type': 'DateDiffV2',
  'params': {'feature': 'lstPmtDate',
   'reference_feature': 'date_of_request',
   'new_feature': 'months_since_lstPmtDate'}},
 {'type': 'PaymentPatternsAggregatorV2',
  'params': {'report_date': 'rptDate',
   'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
     'PAYMENT_HISTORY_1_24',
     'PAYMENT_HISTORY_25_36',
     'PAYMENT_HISTORY_37_48'],
    'rate': {'paid_as_agreed': ['0', '1'],
     'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
     'CO'

In [89]:
preprocess_dict = {}
for preprocessor in node_dict['trade FE'].transformer.transformers['InputNormalizer'].transformers['Preprocessors'].stack:
    name = preprocessor.transformer_name
    print(f'preprocessor: {preprocessor} ')
    print(f'raw_feature input is {preprocessor.input_dtypes}')
    try:
        print(f'output is {preprocessor.new_feature}')
    except:
        print(f'output is {preprocessor.output_dtypes}')
    preprocess_dict[name] = preprocessor


preprocessor: <feature_engine_parts.fe_parts_V2.preprocessors.date_diff.DateDiffV2 object at 0x7fbd41ddcdc0> 
raw_feature input is {'rptDate': 'date', 'date_of_request': 'date'}
output is months_since_rptDate
preprocessor: <feature_engine_parts.fe_parts_V2.preprocessors.date_diff.DateDiffV2 object at 0x7fbd41ddcdf0> 
raw_feature input is {'openDate': 'date', 'date_of_request': 'date'}
output is months_since_openDate
preprocessor: <feature_engine_parts.fe_parts_V2.preprocessors.date_diff.DateDiffV2 object at 0x7fbd41ddce20> 
raw_feature input is {'lstPmtDate': 'date', 'date_of_request': 'date'}
output is months_since_lstPmtDate
preprocessor: <feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator.PaymentPatternsAggregatorV2 object at 0x7fbd41ddce50> 
raw_feature input is {'RATE_STATUS_CODE': 'string', 'PAYMENT_HISTORY_1_24': 'string', 'PAYMENT_HISTORY_25_36': 'string', 'PAYMENT_HISTORY_37_48': 'string', 'months_since_rptDate': 'numeric'}
output is {'number_DQ30+_3_mon

# Preprocessing

After the Mapper converts raw columns to canonical names, the Preprocessor creates derived features and boolean filter flags.



## Date Differences

From the [preprocess asset](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/assets/equifax/cms_6/fe2/trade.json#L261-L285), the first three entries are `DateDiffV2` operations:
```python
{'type': 'DateDiffV2',
 'params': {'feature': 'rptDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_rptDate'}}
```
```python
{'type': 'DateDiffV2',
 'params': {'feature': 'openDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_openDate'}}
```
```python
{'type': 'DateDiffV2',
 'params': {'feature': 'lstPmtDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_lstPmtDate'}}
```
##  Each computes the number of months between `date_of_request` (the reference date from `info`) and the respective date. These tell us how recently the tradeline was reported, how old the account is, and when the last payment was made.


## Payment History Features ([PaymentPatternsAggregatorV2](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py))

This is the most complex preprocessor. The asset entry looks like:
```python
{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    ...
    'DQ30': ['2'], 'DQ60': ['3'], 'DQ90': ['4']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+', 'DQ60+', 'DQ90+', 'DQ120+', 'CO', 'DQ30', 'DQ60', 'DQ90']}}}
```

### Step 1: Construct the zest_payment_pattern ([`_construct_payment_pattern_cols`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L107))
```python
def _construct_payment_pattern_cols(self, data):
    new_ppt = self._combine_payment_patterns(data)
    new_ppt = self._remove_placeholder(new_ppt)
    new_ppt = self._trim(new_ppt)
    new_ppt = self._add_fillers(data, new_ppt)
    return new_ppt

**1a) Combine** all pattern strings left to right ([`_combine_payment_patterns`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L62)):
```python
def _combine_payment_patterns(self, data):
    new_ppt = data[first_ppt_name].copy(deep=True)
    for ppt in self.patterns[1:]:
        ppt_data_ = data[ppt].copy(deep=True)
        ppt_data_.loc[ppt_data_.isna()] = ''
        new_ppt += ppt_data_
    return new_ppt






```


```

This concatenates `RATE_STATUS_CODE` + `PAYMENT_HISTORY_1_24` + `PAYMENT_HISTORY_25_36` + `PAYMENT_HISTORY_37_48` into one string. The `RATE_STATUS_CODE` goes first (leftmost = most recent), then months 1-24, then 25-36, then 37-48.

**1b) Remove placeholders** ([`_remove_placeholder`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L82)): Equifax uses `/` every 12 months as a divider:
```python
def _remove_placeholder(self, new_ppt):
    if self.placeholder:
        new_ppt = new_ppt.str.replace(self.placeholder, '')
    return new_ppt
```

**1c) Trim** to 48 characters so it matches production length ([`_trim`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L91)):
```python
def _trim(self, new_ppt):
    if self.trim:
        new_ppt = new_ppt.str[:self.trim]
    return new_ppt
```

**1d) Add fillers** ([`_add_fillers`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L72)) - prepend `#` characters to the left to account for the gap between the report date and the application date:
```python
def _add_fillers(self, data, new_ppt):
    if self.report_date:
        months_since_ppt = np.ceil(data[self.date_delta_col]).fillna(0).astype(int)
        z = pd.Series(['#']*len(months_since_ppt), index=data.index, dtype='str')
        filler = pd.Series(z).str.repeat(repeats=months_since_ppt.astype(int)).astype('str')
        new_ppt = filler.str.cat(new_ppt, join='left')
    return new_ppt
```

Note that `self.date_delta_col` is `months_since_rptDate` (set from `report_date: 'rptDate'` in the asset). So if the tradeline was reported 2 months before the application date, we prepend `##` to the front. This way position 0 always represents the application month.

### Step 2: Compute payment_history_length ([`_get_effective_month_range`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L98))

Not all characters are valid history. `#` means "before reporting started" and `*` means "break in coverage." Called in `transform`:
```python
data[self.ppt_len_name] = self._get_effective_month_range(
    data[self.zest_ppt_name], 
    data[self.zest_ppt_name].str.len(), 
    exclude_trailing=['*'])
```
```python
def _get_effective_month_range(self, trimmed, month_range, exclude_trailing=[]):
    z_count = trimmed.str.count('#')
    if len(exclude_trailing) > 0:
        z_count = z_count + trimmed.apply(
            lambda x: self._count_trailing_matches(str(x), exclude_trailing))
    effective_month_range = month_range - z_count
    return effective_month_range
```

So: `payment_history_length = total_string_length - #_count - trailing_*_count`

### Step 3: Construct trended features ([`_construct_trended_features`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L116))

For each time window `[3, 6, 12, 24, 48]` months, we trim the string to that window and count delinquency occurrences:
```python
def _construct_trended_features(self, data, new_ppt):
    for month_range in self.month_ranges:
        trimmed = new_ppt.str[:month_range]
        effective_month_count = self._get_effective_month_range(trimmed, month_range)
        for rate, values in self._rate.items():
            count = self._get_count(trimmed, values)
            data[f'number_{rate}_{month_range}_months'] = count.astype('float32')
            data[f'percent_{rate}_{month_range}_months'] = (count/effective_month_count).astype('float32')
```

Where [`_get_count`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L113) counts how many characters in the trimmed string match any of the rate's values:
```python
def _get_count(self, trimmed, values):
    count = trimmed.str.count(values[0])
    for val in values[1:]:
        count += trimmed.str.count(val)
    return count
```

For example with `DQ30+: ['2','3','4','5','6','7','8','9','G','K','L','Z']` and `month_range=12`, it trims the string to the first 12 characters, counts how many match any of those values, and produces `number_DQ30+_12_months` and `percent_DQ30+_12_months` (count divided by effective months in that window).

### Step 4: Construct months_since features ([`_construct_months_since_features`](https://github.com/Katlean/feature-engine-parts/blob/master/feature_engine_parts/fe_parts_V2/preprocessors/payment_pattern_aggregator.py#L125))

For each rate level, find the position of the most recent occurrence in the full payment pattern string:
```python
def _construct_months_since_features(self, data, new_ppt):
    for rate, values in self.rate.items():
        dfs = []
        for v in values:
            loc = data[self.zest_ppt_name].str.find(v).replace(-1, np.nan)
            dfs.append(loc)
        data[f'months_since_most_recent_{rate}'] = pd.concat(dfs, axis=1).min(axis=1)
```

For each value character in that rate, `str.find` returns the first (leftmost) index where it appears, or -1 if not found. We replace -1 with NaN so it doesn't interfere with the `min`. Then we take the minimum across all value characters - that gives us the position of the most recent occurrence.

Since position 0 is the most recent month, the index directly gives the months since. Example: if the string is `##11132111...` and we're looking for `3` (DQ60), `str.find('3')` returns 5, so `months_since_most_recent_DQ60+ = 5`.

Note: this step uses `self.rate` (all rates including `paid_as_agreed`) not `self._rate` (filtered by `keep`), so it produces months_since for all rate categories even though the trended count/percent features in Step 3 only use the `keep` subset.

## Coalescing and Dynamic Placeholders

After payment history, several cleanup steps run. From the [preprocess asset](https://github.com/Katlean/model-engine/blob/06c863fd3dfd76f992584a28da0ae3ad01122876/model_engine/assets/equifax/cms_6/fe2/trade.json):

**CoalesceV2** - if `high_credit_amt` is NA, fill with `chargeoff_amt`:
```python
{'type': 'CoalesceV2',
 'params': {'feature_1': 'high_credit_amt', 'feature_2': 'chargeoff_amt',
  'new_feature': 'high_credit_amt', 'mode': 'na', 'input_dtype': 'numeric'}}
```
The borrower must have reached at least that chargeoff amount in debt or the charge off could not have happened.

**DynamicPlaceholderV2** - set `termDur` to null for revolving/open/charge card accounts:
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'termDur', 'new_value': None,
  'filter_params': {'conditions': [
    {'column': 'PORTFOLIO_TYPE', 'op': 'in', 'value': ['R', 'O', 'C']},
    {'column': 'accountType', 'op': 'in', 'value': ['07', '0G', '18', '2A']}],
   'outer_op': 'any'},
  'raw_feature_input_dtype': 'numeric'}}
```
Loan duration only makes sense for installment loans with fixed repayment schedules.

**DynamicPlaceholderV2** - set `credit_limit` to null for non-revolving accounts:
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'credit_limit', 'new_value': None,
  'filter_params': {'conditions': [
    {'column': 'PORTFOLIO_TYPE', 'op': 'not_in', 'value': ['R', 'O', 'C']},
    {'column': 'accountType', 'op': 'not_in', 'value': ['07', '0G', '18', '2A']}],
   'outer_op': 'all'},
  'raw_feature_input_dtype': 'numeric'}}
```
Credit limit only makes sense for revolving accounts.

**DynamicPlaceholderV2** - set `credit_limit` to null when it's 0 for credit card accounts (likely bad data):
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'credit_limit', 'new_value': None,
  'raw_feature_input_dtype': 'numeric',
  'filter_params': {'conditions': [
    {'column': 'credit_limit', 'op': '==', 'value': 0},
    {'column': 'accountType', 'op': 'in', 'value': ['07', '0G', '18', '2A']}],
   'outer_op': 'all'}}}
```

## Bivariate Ratios

A series of `BivariateComputeV2` operations compute ratios. For example:
```python
{'type': 'BivariateComputeV2',
 'params': {'feature_1': 'balance_amt', 'feature_2': 'credit_limit',
  'op': 'divide', 'new_feature': 'crdUtl'}}
```

The full set:
- `blnc_to_hc` = balance / high_credit - how current debt compares to max exposure
- `hc_to_cl` = high_credit / credit_limit - max historical utilization
- `crdUtl` = balance / credit_limit - current utilization
- `pastDue_to_cl` = past due / credit_limit - severity of delinquency relative to trust extended
- `pastDue_to_hc` = past due / high_credit - severity relative to max exposure
- `pastDue_to_blnc` = past due / balance - severity relative to current debt
- `monthly_payment_amt` = scheduled_payment * termFreqMult - monthly payment amount
- `loan_duration` = termDur * termFreqMult - total loan duration in months
- `months_until_maturity` = loan_duration - months_since_openDate - how many months left
- `acctAge_to_loanDuration` = months_since_openDate / loan_duration - how far along the loan is
- `pastDue_to_monthlyPmt` = past due / monthly payment - how many months of payments are overdue

For `acctAge_to_loanDuration`, values > 1 with positive balance are set to null:
```python
{'type': 'DynamicPlaceholderV2',
 'params': {'raw_feature': 'acctAge_to_loanDuration', 'new_value': None,
  'filter_params': {'conditions': [
    {'column': 'acctAge_to_loanDuration', 'op': '>', 'value': 1},
    {'column': 'balance_amt', 'op': '>=', 'value': 0}],
   'outer_op': 'all'},
  'raw_feature_input_dtype': 'numeric'},
 'notes': "for open_installment loans, this could be bad info or a sign of late pmts..."}
```

If the loan is past its duration but still has a balance, it could be bad data or a sign of late payments. Since we want to monotonically constrain this feature (more age = less risky), we null out these cases so they don't get treated as low risk.

## Filter Flags (Boolean Indicators)

The bulk of preprocessing creates `FilterV2` boolean columns (1 or 0) that the AggregationEngine uses to define which tradelines to group over. For example:
```python
{'type': 'FilterV2',
 'params': {'new_feature': 'is_never_dq',
  'conditions': [
    {'column': 'dqDate', 'op': 'is_na', 'value': None},
    {'column': 'number_DQ30+_48_months', 'op': '==', 'value': 0},
    {'column': 'is_non_derog', 'op': '==', 'value': 1},
    {'column': 'is_deferred', 'op': 'not_in', 'value': [1]},
    {'column': 'RATE_STATUS_CODE', 'op': 'not_in',
     'value': ['2','6','8','3','9','4','G','M','#','7','Z','5']}],
  'outer_op': 'all'}}
```

This checks ALL conditions: no historic dqDate, zero DQ30+ in payment history, currently non-derogatory, not deferred, and rate status is clean. Filters chain together - `is_non_derog` is itself a filter defined earlier that checks `RATE_STATUS_CODE in ['1','2']` AND `is_deferred != 1` AND `is_derog != 1`.

The filters fall into these categories:

**Account status**: `is_open`, `is_closed`, `is_open_active`, `is_transfer`, `is_deferred`, `is_settled`, `is_member`, `has_recent_payment`

**Delinquency status**: `is_dq30`, `is_dq60`, `is_dq90`, `is_dq120+`, `is_derog`, `is_dq`, `is_non_derog`, `is_never_dq`, plus time-windowed versions like `is_dq30+_in_last_24_months`, `is_CO_in_last_12_months`, etc.

**Bankruptcy**: `is_bnkr`, `is_bnkr_ch7`, `is_bnkr_ch13`, `is_bnkr_discharged`

**Account type**: `is_auto`, `is_personal`, `is_credit_card`, `is_charge_card`, `is_homeequity`, `is_mortgage`, `is_first_mortgage`, `is_education`, `is_businessloan`, etc.

**Portfolio type**: `is_installment`, `is_revolving`, `is_loc`, `is_open_ended`, `is_mortgage`

**Utilization buckets**: `has_utilization_over_25_percent`, `has_utilization_over_50_percent`, `has_utilization_over_100_percent` (same for historic utilization using `hc_to_cl`)

**Ownership**: `is_individual`, `is_joint`, `is_joint_not_mortgage`

**Negative events**: `is_chargeoff`, `is_negative`, `is_closed_negative`, `is_foreclosure`, `is_reposession_surrender`, `is_collections`

## Row Dropping

At the end, three `RowDropperV2` operations remove tradelines where the person is not financially responsible:
```python
{'type': 'RowDropperV2',
 'params': {'transformer_name': 'drop_authorized',
  'conditions': [{'column': 'ecoa', 'op': 'in', 'value': ['A']}],
  'outer_op': 'any'},
 'notes': 'dropping authorized trades'}
```

- `ecoa == 'A'` (authorized user - not their account)
- `ecoa == 'T'` (terminated - no longer associated)
- `ecoa == 'X'` (deceased)